# Radiate Traces from Selected Reactome Nodes


This notebook generates trace graphs from nodes selected in the radiate analysis workbook created by `radiate_analysis.ipynb`.

**Before you run this notebook**

- Run `radiate_analysis.ipynb` first.
- Copy the edited selection workbook into `notebooks/input/`.
- In the workbook, set `select = 1` for rows you want to trace on the `pageranks` and/or `reverse pageranks` sheets.
- Confirm your `.env` file contains the Neo4j connection variables required by the first notebook.

**What this notebook writes**

- `output/Radiate_traces_for_<source_name>.graph`: a Sankey-compatible graph file containing the selected forward and reverse traces.

**Tip**

Selections are matched primarily by `stId`. If a selected row has no `stId`, the notebook falls back to the Reactome `dbId` column from the workbook.


In [ ]:
import os
import warnings
from pathlib import Path

import pandas as pd

# Hide noisy warnings so the notebook output stays reader-friendly.
warnings.filterwarnings('ignore')


## Settings


In [ ]:
import dotenv

dotenv.load_dotenv()

required_env_vars = [
    'NEO4J_URI',
    'NEO4J_USERNAME',
    'NEO4J_PASSWORD',
    'NEO4J_DATABASE',
]
missing_env_vars = [name for name in required_env_vars if not os.getenv(name)]
if missing_env_vars:
    raise ValueError(
        'Missing required environment variables: ' + ', '.join(missing_env_vars)
    )

input_dir = Path('input')
output_dir = Path('output')
os.makedirs(output_dir, exist_ok=True)

print(f'Input directory: {Path(input_dir).resolve()}')
print(f'Output directory: {Path(output_dir).resolve()}')


In [ ]:
# Update these values for a new dataset.
source_file = 'als_related_6_genes.csv'
source_name = 'ALS_related_gene_rxns'
nodes_select_file = 'Radiate_analysis_for_ALS_related_gene_rxns_neo4j_selection_2.xlsx'

source_path = input_dir / source_file
selection_path = input_dir / nodes_select_file

for required_file in [source_path, selection_path]:
    if not required_file.exists():
        raise FileNotFoundError(f'Required input file not found: {required_file}')

print(f'Reading source stIds from {source_path}')
print(f'Reading trace selections from {selection_path}')


## Get Source and Target Nodes


In [ ]:
# `sep=None` lets pandas infer comma- vs tab-delimited text files.
df = pd.read_csv(source_path, sep=',', engine='python')
if 'stId' not in df.columns:
    raise ValueError("Input file must include a column named 'stId'.")

st_ids = df['stId'].dropna().astype(str).drop_duplicates().tolist()
if not st_ids:
    raise ValueError('No source stIds were found in the input file.')

print(f'Loaded {len(st_ids)} unique source stIds')
print(st_ids)


In [ ]:
from openpyxl import load_workbook


def get_visible_selected_rows(file_path, sheet_name, select_column='select'):
    """Return rows that are visible in Excel and explicitly marked for tracing."""
    df = pd.read_excel(file_path, sheet_name=sheet_name)
    required_columns = {'dbId', 'name', 'stId', select_column}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(
            f"Sheet '{sheet_name}' is missing required columns: {sorted(missing_columns)}"
        )

    wb = load_workbook(file_path, data_only=True)
    ws = wb[sheet_name]

    visible_df_idx = [
        excel_row - 2
        for excel_row in range(2, ws.max_row + 1)
        if not ws.row_dimensions[excel_row].hidden
    ]

    visible_df = df.iloc[visible_df_idx].copy()
    return visible_df[visible_df[select_column] == 1].copy()


def get_selected_node_ids(file_path, sheet_name, select_column='select'):
    selected_rows = get_visible_selected_rows(file_path, sheet_name, select_column)

    selected_stids = [st_id for st_id in selected_rows['stId'] if pd.notna(st_id)]
    selected_dbids = [
        str(int(node_id))
        for node_id in selected_rows.loc[selected_rows['stId'].isna(), 'dbId']
        if pd.notna(node_id)
    ]

    if selected_dbids:
        print('Selected rows without stId; falling back to Reactome dbIds:')
        print(selected_rows.loc[selected_rows['stId'].isna(), ['dbId', 'stId', 'name']])

    print(f'Selected {len(selected_stids)} stIds from {sheet_name}')
    print(f'Selected {len(selected_dbids)} dbIds from {sheet_name}')
    return {'stIds': selected_stids, 'dbIds': selected_dbids}


forward_selection = get_selected_node_ids(selection_path, 'pageranks')
reverse_selection = get_selected_node_ids(selection_path, 'reverse pageranks')


## Connect to Neo4j and Resolve the Selected Nodes


In [ ]:
from lifelike_gds.graph_sources import Reactome, ReactomeDB
from lifelike_gds.graph_sources.domain_config import REACTOME_TRACE_NODE_LABEL
from lifelike_gds.network.radiate_trace import RadiateTrace


def _get_database() -> ReactomeDB:
    return ReactomeDB(
        uri=os.getenv('NEO4J_URI'),
        username=os.getenv('NEO4J_USERNAME'),
        password=os.getenv('NEO4J_PASSWORD'),
        database=os.getenv('NEO4J_DATABASE'),
    )


def get_nodes_from_selection(database, selection):
    """Resolve selected workbook rows to Reactome nodes using stId first, then dbId."""
    nodes = []

    if selection['stIds']:
        nodes.extend(
            database.get_nodes_by_attr(
                attr_values=selection['stIds'],
                attr_name='stId',
                node_label=REACTOME_TRACE_NODE_LABEL,
            )
        )

    if selection['dbIds']:
        nodes.extend(
            database.get_nodes_by_attr(
                attr_values=selection['dbIds'],
                attr_name='dbId',
                node_label=REACTOME_TRACE_NODE_LABEL,
            )
        )

    # Deduplicate while preserving order so repeated selections do not create duplicate traces.
    unique_nodes = []
    seen_ids = set()
    for node in nodes:
        node_id = node.get('id')
        if node_id in seen_ids:
            continue
        seen_ids.add(node_id)
        unique_nodes.append(node)
    return unique_nodes


def export_radiate_traces(
    tracegraph,
    source_name,
    source_nodes,
    forward_nodes: list | None = None,
    reverse_nodes: list | None = None,
):
    """Export forward and reverse radiate traces for the selected nodes."""
    tracegraph.graph = tracegraph.orig_graph.copy()
    tracegraph.set_node_set_from_db_nodes(source_nodes, source_name, source_name)

    pagerank_prop = 'pagerank'
    rev_pagerank_prop = 'rev_pagerank'

    if forward_nodes:
        tracegraph.set_pagerank(source_name, pagerank_prop, False)

    if reverse_nodes:
        tracegraph.set_pagerank(source_name, rev_pagerank_prop, True)

    tracegraph.add_graph_description('Reactome')

    if forward_nodes:
        nodeset_name = 'forward select'
        tracegraph.set_node_set_from_db_nodes(
            forward_nodes, nodeset_name, nodeset_name
        )
        tracegraph.add_traces_from_sources_to_each_selected_nodes(
            forward_nodes,
            source_name,
            weighted_prop=pagerank_prop,
            selected_nodes_name=nodeset_name,
        )
        tracegraph.add_trace_from_sources_to_all_selected_nodes(
            nodeset_name,
            source_name,
            weighted_prop=pagerank_prop,
            trace_name='Forward combined selected nodes',
        )

    if reverse_nodes:
        nodeset_name = 'reverse select'
        tracegraph.set_node_set_from_db_nodes(
            reverse_nodes, nodeset_name, nodeset_name
        )
        tracegraph.add_traces_from_each_selected_nodes_to_targets(
            reverse_nodes,
            source_name,
            weighted_prop=rev_pagerank_prop,
            selected_nodes_name=nodeset_name,
        )
        tracegraph.add_trace_from_all_selected_nodes_to_targets(
            nodeset_name,
            source_name,
            weighted_prop=rev_pagerank_prop,
            trace_name='Reverse combined selected nodes',
        )

    graph_file = f'Radiate_traces_for_{source_name}.graph'
    tracegraph.write_to_sankey_file(graph_file)
    return output_dir / graph_file


In [ ]:
database = _get_database()
graphsource = Reactome(database)

source_nodes = database.get_nodes_by_attr(
    attr_values=st_ids,
    attr_name='stId',
    node_label=graphsource.get_node_query_label(),
)
if not source_nodes:
    raise ValueError('No source nodes were found for the provided source stIds.')

forward_nodes = get_nodes_from_selection(database, forward_selection)
reverse_nodes = get_nodes_from_selection(database, reverse_selection)

if not forward_nodes and not reverse_nodes:
    raise ValueError(
        'No trace targets were selected. Mark rows with select = 1 in the workbook first.'
    )

print(
    f'Found {len(source_nodes)} source nodes, '
    f'{len(forward_nodes)} forward nodes, and '
    f'{len(reverse_nodes)} reverse nodes.'
)


## Load the Reactome Graph into Memory


In [ ]:
tracegraph = RadiateTrace(Reactome(database))

# Write notebook outputs to the shared `notebooks/output/` directory.
tracegraph.datadir = output_dir

# Build the in-memory NetworkX graph used for tracing.
tracegraph.init_default_graph()


## Export Selected Traces


In [ ]:
trace_file = export_radiate_traces(
    tracegraph,
    source_name,
    source_nodes,
    forward_nodes=forward_nodes,
    reverse_nodes=reverse_nodes,
)
print(f'Radiate trace graph exported to {trace_file.resolve()}')


## Notes

The exported `.graph` file can be opened by downstream Sankey tooling. Re-run this notebook whenever you update the workbook selections to regenerate the trace graph.
